# Zero-day novelty detection: the original design

Normal-only training: the detector is fitted on 70% of normal traffic and scored
on the other 30% plus **every** attack row, so every attack type is unseen. The
detector is the linear one (Mahalanobis in a PCA subspace + reconstruction
residual) at every reservoir size, as in the original sweep
(config: `configs/novelty_reference.yaml`).

| original name | here |
|---|---|
| NOV_q | quantum x pca32 |
| NOV_dials | dials x pca32 |
| NOV_raw | raw x pca32 |
| IForest | raw x iforest |

Headline metrics: `macro_auc` (attack types with >= 20 rows weighted equally --
pooled AUC is ~70% neptune) and `det@0.01`, detection at a 1% false-alarm budget,
the number that matters when deploying.

In [ ]:
# Setup: make the repo importable from notebooks/ (or from the repo root).
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
REPO = REPO.parent if REPO.name == "notebooks" else REPO
sys.path.insert(0, str(REPO))

import pandas as pd
from elements import config
from elements.eval.results import load_results
from elements.experiments.runner import plan, run
from elements.viz import scaling
pd.set_option("display.precision", 4)
pd.set_option("display.width", 160)
from elements.viz import novelty as nv

In [ ]:
CFG = config.load(REPO / "configs" / "novelty_reference.yaml")
FIGS = CFG.out_dir / "figs"
print(plan(CFG))

### Run

Off by default so opening the notebook never starts a long run. Set `RUN = True`
(or run `python -m elements.run configs/novelty_reference.yaml` from the repo root). Every trained head is
appended to `results.jsonl` as it finishes, so an interrupted run resumes where
it stopped.

In [ ]:
RUN = False
if RUN:
    run(CFG)

## Results

In [ ]:
DF = load_results(CFG.out_dir, keep=("per_type",))
scaling.capacity(DF, "macro_auc", head="pca32", arms=("quantum", "dials"),
                 out=FIGS / "capacity_macro_auc.png");
scaling.capacity(DF, "det@0.01", head="pca32", arms=("quantum", "dials"),
                 out=FIGS / "capacity_det1.png");

In [ ]:
nv.detection_table(DF)

### Which zero-days are catchable?

In [ ]:
nv.per_attack_heatmap(DF, "quantum", "pca32", out=FIGS / "per_attack_quantum.png");

In [ ]:
nv.per_attack(DF, "raw", "iforest").round(3)

**Caveats.** One seed. AUC is threshold-free and flatters a detector that would
be useless in practice -- read the `det@` columns. NSL-KDD attacks are known
attacks the detector was not shown, not genuinely novel traffic.